In [ ]:
#%%apyter init
from appyter import magic
magic.init(lambda _=globals: _())

In [ ]:
%%appyter hide_code

{% do SectionField(
    name='input', 
    title = 'Gene of interest', 
    subtitle = 'Enter a gene for which you wish to get up and down regulators.'
) %}

In [ ]:
%%appyter hide_code

{% set input_gene = AutocompleteField(
    name = 'input_gene',
    label = 'Query Gene',
    default = 'C9ORF72',
    description = 'Enter the gene symbol of interest.',
    file_path = 'https://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/all_genes.json',
    section='input'
)%}

In [ ]:
%%appyter code_exec
query_gene = "{{ input_gene.value.upper() }}"

# Drug Gene Budger 2

This notebook takes a gene as input and identifies drugs that maximally up and down regulate the gene's expression in a collection of chemical perturbation datasets.

- Ginkgo GDPx1 and GPDx2: Limma-Voom based differential gene epxression results for 1,354 drugs.
- Novartis DRUG-seq: Differential: Limma-Trend based differential expression results for 4,343 drugs. 
- LINCS L1000 Chemical Perturbations: Queries the [LINCS Reverse Search Dashboard](https://lincs-reverse-search-dashboard.dev.maayanlab.cloud/) for pre-computed characteristic direction-based differential gene expression signatures from RNA-seq-like LINCS L1000 Expression Profiles covering 33,571 drugs.

The Ginkgo dataset includes 4 primary cell types (eithelial melanocytes, smooth aortic muscle cells, skeletal muscle myoblasts and dermal fibroblasts) and one cell line (A549 lung carcinoma cell line). Previous analysis showed distinct transcriptional responses by cell type, so the drug rankings for the Ginkgo dataset are separated by cell type.

In [ ]:
## General
import pandas as pd
import numpy as np
import re
import warnings

## HTTP Requests
import requests

## Tables
from IPython.display import display, display_markdown, HTML

## UpSet Plot
from upsetplot import from_contents, plot
from matplotlib import pyplot

## Volcano Plot
from bokeh.plotting import figure, show
from bokeh.models import ColumnDataSource, HoverTool, LinearColorMapper
from bokeh.palettes import RdBu
from bokeh.io import output_notebook

In [ ]:
# Storage url for Ginkgo and Novartis DE files
ginkgo_URL = 'http://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/ginkgo_de'
novartis_URL = 'http://appyters.maayanlab.cloud/storage/DrugRegulators_Appyter/novartis_de'
# silence warnings
warnings.filterwarnings('ignore')

In [ ]:
in_ginkgo = in_novartis = in_lincs = True

In [ ]:
# get Ginkgo DE results for gene
gene_file = f'{query_gene}.f'
try:
    ginkgo_de = pd.read_feather(f'{ginkgo_URL}/{gene_file}')
    ginkgo_cell_types = list(set(p.split('-')[0] for p in ginkgo_de.Perturbation))
except:
    in_ginkgo=False
    print('Gene not in Ginkgo dataset')
    

In [ ]:
def prepare_ginkgo_data(df, cell_types):
    '''Create a results dictionary where each cell type
    in the Ginkgo dataset is a key and the value is the DE data
    for the query gene for that cell type.
    '''
    # get perturbations with given cell type
    cell_type_results = {}
    for k in cell_types:
        subset = df[df['Perturbation'].str.contains(k)]
        subset['log10adj.P.Val'] = subset['adj.P.Val'].replace(0,1e-323).map(np.log10)*-1
        cell_type_results[k] = subset
    return cell_type_results
    

In [ ]:
if in_ginkgo:
    ginkgo_gene_expr_dict = prepare_ginkgo_data(ginkgo_de, ginkgo_cell_types)

In [ ]:
def l1000_reverse_search(gene_id:str, direction:str):
    url = f'https://lincs-reverse-search-dashboard.dev.maayanlab.cloud/api/table/cp/{direction}/{gene_id}'
    headers = {
        'Accept':'application/json',
        'Content-Type':'application/json'
    }
    try:
        resp = requests.get(url, headers=headers)
        resp.raise_for_status()
        res = resp.json()
        df = pd.DataFrame(res)
    except requests.exceptions.HTTPError as e:
        print(f"Gene not found in LINCS: {e}")
        df=pd.DataFrame()
    return df
    

In [ ]:
l1000_up = l1000_reverse_search(query_gene.upper(), 'up')
l1000_down = l1000_reverse_search(query_gene.upper(), 'down')
if l1000_down.empty or l1000_up.empty:
    in_lincs=False

In [ ]:
# get Novartis DE results for gene
try:
    novartis_de = pd.read_feather(f'{novartis_URL}/{gene_file}').set_index('index')
     # format p-values
    novartis_de['log10adj.P.Val'] = novartis_de['P.Adj'].replace(0,1e-323).map(np.log10)*-1
    # rename logFC column for concordance with Ginkgo columns
    novartis_de.rename(columns={'LogFC':'logFC'}, inplace=True)
except:
    print('Gene not in novartis dataset')
    in_novartis=False

In [ ]:
if in_lincs + in_novartis + in_ginkgo < 2:
    print(f"LINCS: {in_lincs}")
    print(f"Novartis: {in_novartis}")
    print(f"Ginkgo: {in_ginkgo}")
    raise Exception("Execution stopped, gene not found in at least 2 datasets")

## Query Gene

In [ ]:
display_markdown(f"This notebook shows results for the input gene **{query_gene}**", raw=True)

## Rank Drugs

Within each dataset drugs are ranked by the statistical significance of the regulatory relationship. The pipeline uses the differential expression p-value for the Novartis and Ginkgo data and the characterstic direction coefficient for the LINCS L1000 data. 

When a dataset contains multiple perturbations for the same drug (i.e. a cell exposed to the drug at different doses), p-values are averaged across doses to get a single ranking for the drug. 

In [ ]:
def get_rankings(data:pd.DataFrame, source:str, cell_type:str, direction:str):
    '''
    Given a dataframe of logFC and p-values for a gene of interest across perturbations, 
    rank the drugs by how the induce or repress the gene. 

    Returns a tuple of 1) drug ranks averaged across drug dosages and 2) full
    perturbation ranks. 
    '''
    ranked_data = data.copy()
    
    if (source == 'Ginkgo') & (cell_type=='A549'):
        ranked_data.loc[ranked_data['Drug']=='Brefeldin A from Penicillium brefeldianum', 'Drug'] = 'Brefeldin A'
    elif (source == 'Ginkgo') & (cell_type != 'A549'):
        ranked_data.loc[ranked_data['Drug']=='Brefeldin-A', 'Drug'] = 'Brefeldin A'
    elif source == 'Novartis':
        ranked_data.loc[ranked_data['Drug']=='Trichostatin A (racemate)', 'Drug'] = 'Trichostatin A'
    # average rank across all drug dosages
    drug_mean_ranks = ranked_data.loc[:,['Drug','logFC','log10adj.P.Val']].groupby('Drug')[['logFC','log10adj.P.Val']].mean().sort_values('log10adj.P.Val', ascending=False)
    # filter for up or down regulation
    if direction == 'up':
        drug_mean_ranks = drug_mean_ranks[drug_mean_ranks['logFC'] > 0]
    elif direction == 'down':
        drug_mean_ranks = drug_mean_ranks[drug_mean_ranks['logFC'] < 0]
    drug_mean_ranks.rename(columns={'logFC':'Avg logFC', 'log10adj.P.Val':'Avg -log10(Adj.PVal)'}, inplace=True)
    return drug_mean_ranks, ranked_data

def get_top(rank_results:pd.DataFrame, n=50):
    '''
    Given the drug_mean_ranks result from get_rankings, extract the names of the drugs
    that most down- or up-regulate the gene of interest (top N).

    If there are less drugs than N, will return all results.
    '''
    top = {d.casefold() for d in set(rank_results.head(n).index)}
    return top

# create download link for table results
def download_link(df, fname):
    if df.shape[0] == 0: return ''
    csv = df.to_csv(fname, sep='\t', index=True)
    link = f'<div>Download full results: <a href="{fname}" target=_blank>{fname}</a></div>'
    return link

### Ginkgo

Drug rankings for the Ginkgo dataset. Top 20 by p-value are shown and the complete table can be downloaded.

In [ ]:
top_n = 20

In [ ]:
ginkgo_drugs_up = {}
ginkgo_drugs_down = {}
for cell_type, exprdf in ginkgo_gene_expr_dict.items():
    # rank by level of up-regulation
    mean_ranks, full_ranks = get_rankings(exprdf, 'Ginkgo', cell_type, 'up')
    ginkgo_drugs_up[cell_type] = (mean_ranks, full_ranks)
    display_markdown(f'**Top {top_n} up-regulators for {cell_type}**', raw=True)
    display(mean_ranks.head(top_n))
    display(HTML(download_link(mean_ranks, f"ginkgo_drug_ranks_UpReg_{cell_type}.tsv")))
    # rank by level of down-regulation
    mean_ranks, full_ranks = get_rankings(exprdf, 'Ginkgo', cell_type, 'down')
    ginkgo_drugs_down[cell_type] = (mean_ranks, full_ranks)
    display_markdown(f'**Top {top_n} down-regulators for {cell_type}**', raw=True)
    display(mean_ranks.head(top_n))
    display(HTML(download_link(mean_ranks, f"ginkgo_drug_ranks_DnReg_{cell_type}.tsv")))


### L1000

Drug rankings for the LINCS L1000 dataset. Top 20 by CD-coefficient are shown and the complete table can be downloaded.

In [ ]:
def l1000_sort(result, direction):
    df = result[['Perturbagen','CD Coefficient']]
    df = df.groupby('Perturbagen').mean(['CD Coefficient'])
    if direction == 'up':
        df = df.sort_values('CD Coefficient', ascending=False)
    elif direction == 'down':
        df = df.sort_values('CD Coefficient', ascending=True)
    return df

l1000_top_up = l1000_sort(l1000_up, 'up')
l1000_top_down = l1000_sort(l1000_down, 'down')
display_markdown(f'**Top {top_n} up-regulators in L1000**', raw=True)
display(l1000_top_up.head(top_n))
display(HTML(download_link(l1000_top_up, f"l1000_drug_ranks_UpReg.tsv")))
display_markdown(f'**Top {top_n} down-regulators in L1000**', raw=True)
display(l1000_top_down.head(top_n))
display(HTML(download_link(l1000_top_down, f"l1000_drug_ranks_DnReg.tsv")))

### Novartis DRUG-seq

Drug rankings for the Novartis dataset. Top 20 by p-value are shown and the complete table can be downloaded.

In [ ]:
novartis_drugs_up = get_rankings(novartis_de, 'Novartis', '', 'up')
novartis_drugs_down = get_rankings(novartis_de, 'Novartis', '', 'down')

display_markdown(f'**Top {top_n} up-regulators in Novartis DRUG-seq**', raw=True)
display(novartis_drugs_up[0].head(top_n))
display(HTML(download_link(novartis_drugs_up[0], 'novartis_drug_ranks_UpReg.tsv')))
display_markdown(f'**Top {top_n} down-regulators in Novartis DRUG-seq**', raw=True)
display(novartis_drugs_down[0].head(top_n))
display(HTML(download_link(novartis_drugs_down[0], 'novartis_drug_ranks_DnReg.tsv')))

In [ ]:
top_up = {}
top_down = {}
# get results from Ginkgo
for cell_type in ginkgo_drugs_down.keys():
    top_up[f'ginkgo_{cell_type}'] = get_top(ginkgo_drugs_up[cell_type][0], n=50)
    top_down[f'ginkgo_{cell_type}'] = get_top(ginkgo_drugs_down[cell_type][0], n=50)
# get results from L1000
top_up['lincs_l1000'] = {drug.casefold() for drug in set(l1000_top_up.index)}
top_down['lincs_l1000'] = {drug.casefold() for drug in set(l1000_top_down.index)}
# get results from novartis
top_up['novartis'] = get_top(novartis_drugs_up[0], n=50)
top_down['novartis'] = get_top(novartis_drugs_down[0], n=50)

## UpSet Plot

The UpSet plots show the overlap among top up-regulating or down-regulating drugs in each dataset. If there were more than 50 significant regulators in a dataset for a given input gene, the input was restricted to the top 50 regulators.

In [ ]:
# Saving Figures
def save_figure(plot_name, **kwargs):
    import io
    mem = io.BytesIO()
    pyplot.savefig(mem, bbox_inches='tight')
    with open(plot_name, 'wb') as fw:
        fw.write(mem.getbuffer())

In [ ]:
def create_upset(top_sets: dict):
    rename_keys = {
        'ginkgo_A549': 'ginkgo_A549',
        'lincs_l1000': 'lincs_l1000',
        'novartis': 'novartis',
        'ginkgo_human_epithelial_melanocytes': 'ginkgo_melanocytes',
        'ginkgo_human_dermal_fibroblast': 'ginkgo_fibroblasts',
        'ginkgo_human_aortic_smooth_muscle_cells': 'ginkgo_muscle_cells',
        'ginkgo_human_skeletal_muscle_myoblasts': 'ginkgo_myoblasts'
    }
    top_sets = {rename_keys[k]:v for k,v in top_sets.items()}
    upset_data = from_contents(top_sets)
    plot(upset_data, orientation = 'horizontal', show_counts = True, element_size = 30)
    pyplot.show()

In [ ]:
display_markdown(f"**Overlap among top up regulators of {query_gene}**", raw=True)
create_upset(top_up)

In [ ]:
display_markdown(f"**Overlap among top down regulators of {query_gene}**", raw=True)
create_upset(top_down)

In [ ]:
def get_overlapping_sets(top_sets:dict, to_file:str):
    '''
    Given the dictionary of sets used to created the UpSet plot,
    return the contents of the overlapping sets. 
    '''
    # convert to multi-index dataframe
    set_df = from_contents(top_sets)
    multi_index_df = pd.DataFrame(columns=list(top_sets.keys()))
    for colname in multi_index_df.columns:
        multi_index_df[colname] = set_df.index.get_level_values(colname).to_list()
    # only keep unique sets of intersection contributors
    multi_index_df.drop_duplicates(inplace=True)
    # sort multi-index for efficient indexing
    set_df = set_df.sort_index()
    # extract drug intersection for each group
    overlapping_sets = pd.DataFrame(columns=['Members', 'Overlap', 'Length'])
    for idx in range(multi_index_df.shape[0]):
        ixn_drugs = set_df.loc[tuple(multi_index_df.iloc[idx])].id.to_list()
        # get group members
        ixn_name = multi_index_df.iloc[idx][multi_index_df.iloc[idx]].index.to_list()
        ixn_name_joined = '-'.join(ixn_name)
        # append results
        overlapping_sets = pd.concat([overlapping_sets, pd.DataFrame({'Members':ixn_name_joined, 'Overlap':[ixn_drugs], 'Length':len(ixn_drugs), 'N Datasets':len(ixn_name)})])
        
    
    overlapping_sets = overlapping_sets.sort_values('N Datasets', ascending=False)
    return overlapping_sets


Below are tabular representations of the UpSet plots.

In [ ]:
overlap_down = get_overlapping_sets(top_down, 'overlap_down')
overlap_up = get_overlapping_sets(top_up, 'overlap_up')
display_markdown("**Down-regulating drug overlap**", raw=True)
display(overlap_down)
display(HTML(download_link(overlap_down, 'overlapping_drugs_DnReg.tsv')))
display_markdown("**Up-regulating drug overlap**", raw=True)
display(overlap_up)
display(HTML(download_link(overlap_up, 'overlapping_drugs_UpReg.tsv')))


In [ ]:
def get_logFC_averages(overlapping_df, direction):
    '''
    Retrieve average logFC across datasets for drugs in overlapping sets. 

    Returns dataframe with columns for:
    Drug
    Average logFC
    Number of datasets for which drug was a significant regulator of the query gene
    '''
    # extract up or down data
    if direction == 'down':
        data_dict ={
            'ginkgo_A549': ginkgo_drugs_down['A549'][1],
            'ginkgo_human_dermal_fibroblast': ginkgo_drugs_down['human_dermal_fibroblast'][1],
            'ginkgo_human_aortic_smooth_muscle_cells': ginkgo_drugs_down['human_aortic_smooth_muscle_cells'][1],
            'ginkgo_human_epithelial_melanocytes':ginkgo_drugs_down['human_epithelial_melanocytes'][1],
            'ginkgo_human_skeletal_muscle_myoblasts':ginkgo_drugs_down['human_skeletal_muscle_myoblasts'][1],
            'novartis': novartis_drugs_down[1],
            'lincs': l1000_down
        }
    elif direction == 'up':
        data_dict ={
            'ginkgo_A549': ginkgo_drugs_up['A549'][1],
            'ginkgo_human_dermal_fibroblast': ginkgo_drugs_up['human_dermal_fibroblast'][1],
            'ginkgo_human_aortic_smooth_muscle_cells': ginkgo_drugs_up['human_aortic_smooth_muscle_cells'][1],
            'ginkgo_human_epithelial_melanocytes':ginkgo_drugs_up['human_epithelial_melanocytes'][1],
            'ginkgo_human_skeletal_muscle_myoblasts':ginkgo_drugs_up['human_skeletal_muscle_myoblasts'][1],
            'novartis': novartis_drugs_up[1],
            'lincs': l1000_up
        }
    # get average, integrating across datasets
    average_logfc_vals = {}
    n_datasets = list()
    for _,row in overlapping_df.iterrows():
        n_datasets.extend([row['N Datasets']]*len(row['Overlap']))
        for d in row['Overlap']:
            n = 0
            runsum = 0
            for k,df in data_dict.items():
                if k == 'lincs':
                    subset = df[df['Perturbagen'].str.lower() == d.lower()]
                    subset.rename(columns = {'Log2(Fold Change)':'logFC'}, inplace=True)
                else:
                    subset = df[df['Drug'].str.lower() == d.lower()]
                n = n + subset.shape[0]
                runsum = runsum + subset.logFC.sum()
            average_logfc_vals[d] = round(runsum / n,3)
    # create results dataframe
    res_df = pd.DataFrame({
        'Drug': list(average_logfc_vals.keys()),
        'Avg(logFC)': list(average_logfc_vals.values())
    })
    res_df['N Datasets'] = n_datasets
    # sort based on N datasets and logFC, direction depending on up or down set
    if direction=='up':
        res_df = res_df.sort_values(['N Datasets','Avg(logFC)'], ascending=[False,False])
    elif direction == 'down':
        res_df = res_df.sort_values(['N Datasets','Avg(logFC)'], ascending=[False,True])
    return res_df

            
overlapping_up_logFC = get_logFC_averages(overlap_up, 'up')
overlapping_down_logFC = get_logFC_averages(overlap_down, 'down')

Tables that show the average logFC value (across datasets) for drugs that were found to be significant regulators in more than one dataset.

In [ ]:
display_markdown("**Average LogFC values across datasets: Up-regulating drugs**", raw=True)
display(overlapping_up_logFC.head(n=top_n))
display(HTML(download_link(overlapping_up_logFC, 'overlapping_drugs_logfc_UpReg.tsv')))
display_markdown("**Average LogFC values across datasets: Down-regulating drugs**", raw=True)
display(overlapping_down_logFC.head(n=top_n))
display(HTML(download_link(overlapping_down_logFC, 'overlapping_drugs_logfc_DnReg.tsv')))


## Volcano Plots

The volcano plots show the strength and statistical significance of the drug perturbation for each signature in the dataset (drug and dose specific). Color of points indicate up (red) or down (blue) regulation. Hover over points in the volcano plot to see the label (with cell type, drug, and dose information), logFC, fold-change, and log10-transformed p-value or CD-coefficient. Tools to the right of the plot allow you to manipulate (pan, zoom) and download the figure. 

In [ ]:
output_notebook()

def create_bokeh_volcano_plot(expr_data:pd.DataFrame, gene_id:str, cell_type:str, source:str):
    '''
    Given the expression data for a given gene, create an interactive
    volcano plot that shows regulation of gene across all perturbations (drug, dosage, cell line).
    '''
    
    df = expr_data.copy()
    
    # clean columns
    if source == 'Ginkgo':
        df['Label'] = df['Perturbation']
        df['FC'] = 2**df['logFC']
    elif source == 'Novartis':
        df['Label'] = df['Perturbation'] + '_' + df['Drug']
        df['FC'] = 2**df['logFC']
    elif source == 'L1000':
        df.rename(columns={'Perturbagen':'Drug', 'Log2(Fold Change)':'logFC','Fold Change':'FC'}, inplace=True)
        df['absCDcoef'] = abs(df['CD Coefficient'])
        df['Label'] = df.index.to_list()

    # set plot source
    if source == 'Ginkgo' or source == 'Novartis':
        plot_source = ColumnDataSource(df.loc[:,['Label','logFC','FC','log10adj.P.Val']])
        x,y='logFC','log10adj.P.Val'
        hover = HoverTool(tooltips=[("Label", "@Label"),
                                ("Log2(FC)", "@logFC"),
                                ("Fold Change", "@FC"),
                                ('-Log10(Adj. P-value)',"@{log10adj.P.Val}{0.00e}")])
    elif source=='L1000':
        plot_source = ColumnDataSource(df.loc[:,['Label','logFC','FC','absCDcoef']])
        x,y = 'logFC','absCDcoef'
        hover = HoverTool(tooltips=[("Label", "@Label"),
                                ("Log2(FC)", "@logFC"),
                                ("Fold Change", "@FC"),
                                ('abs(CD Coefficient)',"@{absCDcoef}{0.00e}")])
        
    # define figure
    p = figure(
        title=f'{gene_id} Regulation in {source} {cell_type}',
        x_axis_label = 'Log2(Fold Change)',
        y_axis_label = 'abs(CD Coefficient)' if source == 'L1000' else '-Log10(Adj. P-value)',
        tools = 'pan,wheel_zoom,box_zoom,reset,save'
    )

    # color mapper
    color_mapper = LinearColorMapper(palette = RdBu[10],
                                     low = min(df['logFC']),
                                     high=max(df['logFC']))
    # plot
    p.scatter(x=x,
              y=y,
              size=8,
              source=plot_source,
              fill_alpha=0.6,
              color = {'field':'logFC','transform':color_mapper})
    p.add_tools(hover)
    show(p)

### Ginkgo

In [ ]:
for cell_type, expr_df in ginkgo_gene_expr_dict.items():
    cell_name = ' '.join(re.sub('human_','',cell_type).split('_'))
    create_bokeh_volcano_plot(expr_df, query_gene, cell_name, 'Ginkgo')

### L1000

In [ ]:
create_bokeh_volcano_plot(pd.concat([l1000_up,l1000_down]), query_gene, '', 'L1000')

### Novartis DRUG-seq

In [ ]:
create_bokeh_volcano_plot(novartis_de, query_gene, '', 'Novartis')